# Visão Computacional — Pré-processamento de Imagens
### Do ruído à forma: filtros, bordas e segmentação

**Tópicos:**
1. Ruídos em Imagens
2. Suavização (Filtros de Média e de Ordem)
3. Limiarização (Global, Otsu e Adaptativa)
4. Operações Morfológicas (Erosão e Dilatação)
5. Detecção de Bordas (Sobel, Laplaciano e Canny)
6. Segmentação (Crescimento de Região, Componentes Conectados, Contornos, Casca Convexa)
7. Case Prático: Manchas de Óleo
8. Exercícios Práticos
9. Bônus: Esteganografia

> **Pré-requisito:** rode as células de setup antes de executar qualquer outra seção.

## Setup — Dependências e repositório

In [ ]:
# Clona o repositório com as imagens utilizadas na aula
!rm -rf fiap-ml-visao-computacional/
!git clone https://github.com/FIAPON/fiap-ml-visao-computacional.git
%cd fiap-ml-visao-computacional/aula-3-segmentacao/

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import random
import os
import pandas as pd
from glob import glob
from scipy.ndimage import generic_filter, minimum_filter, maximum_filter
from PIL import Image

%matplotlib inline
sns.set_style("whitegrid", {'axes.grid': False})

# Função utilitária para exibir grade de imagens
def mostrar_grade(imagens, titulos, cmap='gray', figsize=(18, 5)):
    n = len(imagens)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, titulo in zip(axes, imagens, titulos):
        ax.imshow(img, cmap=cmap)
        ax.set_title(titulo, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("Ambiente configurado com sucesso.")

---
## 1. Ruídos em Imagens

Ruído é a **variação indesejada da intensidade dos pixels**, modelada como:

> g(x,y) = f(x,y) + n(x,y)

onde `f` é a imagem original e `n` é o componente de ruído.

**Principais fontes:**
- Ruído térmico em sensores CMOS/CCD (agitação dos elétrons)
- Interferência eletromagnética (ambientes industriais/hospitalares)
- Compressão e transmissão de dados (artefatos JPEG)
- Processos estocásticos naturais (shot noise quântico)

| Tipo | Aplicação típica | Característica |
|------|-----------------|----------------|
| Gaussiano | Câmeras médicas, inspeção industrial | Distribuição normal N(μ,σ²); afeta todos os pixels |
| Rayleigh | LIDAR, ultrassom, radar SAR | PDF assimétrica; ruído sempre positivo (speckle) |
| Erlang (Gama) | Tomografia PET | Caso especial Gama com parâmetro inteiro |
| Exponencial | Fluorescência, sensores ópticos | Cauda longa; mais agressivo em regiões escuras |
| Impulsivo (Sal e Pimenta) | Sensores danificados | Pixels assumem 0 ou 255; filtro de mediana é ideal |

In [ ]:
# Funções de geração de ruído

def add_gaussian_noise(image, mean=0, std=50):
    gauss = np.random.normal(mean, std, image.shape).astype(np.float32)
    return np.clip(image.astype(np.float32) + gauss, 0, 255).astype(np.uint8)

def add_rayleigh_noise(image, scale=60):
    rayleigh = np.random.rayleigh(scale, image.shape).astype(np.float32)
    return np.clip(image.astype(np.float32) + rayleigh, 0, 255).astype(np.uint8)

def add_erlang_noise(image, shape=3, scale=50):
    erlang = np.random.gamma(shape, scale, image.shape).astype(np.float32)
    return np.clip(image.astype(np.float32) + erlang, 0, 255).astype(np.uint8)

def add_exponential_noise(image, scale=80):
    exp = np.random.exponential(scale, image.shape).astype(np.float32)
    return np.clip(image.astype(np.float32) + exp, 0, 255).astype(np.uint8)

def add_impulse_noise(image, amount=0.10, salt_vs_pepper=0.5):
    noisy = np.copy(image)
    num_salt   = np.ceil(amount * image.size * salt_vs_pepper).astype(int)
    num_pepper = np.ceil(amount * image.size * (1.0 - salt_vs_pepper)).astype(int)
    coords = [np.random.randint(0, i - 1, num_salt) for i in image.shape]
    noisy[tuple(coords)] = 255
    coords = [np.random.randint(0, i - 1, num_pepper) for i in image.shape]
    noisy[tuple(coords)] = 0
    return noisy

In [ ]:
# Visualização comparativa dos tipos de ruído
original = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)

ruidos = {
    "Original":    original,
    "Gaussiano":   add_gaussian_noise(original),
    "Rayleigh":    add_rayleigh_noise(original),
    "Erlang":      add_erlang_noise(original),
    "Exponencial": add_exponential_noise(original),
    "Impulsivo":   add_impulse_noise(original),
}

mostrar_grade(list(ruidos.values()), list(ruidos.keys()), figsize=(20, 4))

---
## 2. Suavização de Imagens

Para eliminar ruídos aplicamos **filtros de suavização** por meio de convolução espacial:

> (f * k)(x,y) = Σ Σ f(x-i, y-j) · k(i,j)

O **kernel** é uma matriz de pesos que define a contribuição de cada pixel vizinho.
Kernels maiores → suavização mais intensa → custo computacional O(n²) por pixel.

> **Regra prática:** sempre use dimensão ímpar (3×3, 5×5, 7×7) para garantir pixel central definido.

### 2.1 Filtros de Média

In [ ]:
# Implementação dos filtros de média─

def filtro_media_aritmetica(img, kernel_size=3):
    """Substitui cada pixel pela média aritmética da vizinhança. Sensível a outliers."""
    return cv2.blur(img, (kernel_size, kernel_size))

def filtro_media_geometrica(img, kernel_size=3):
    """Preserva melhor os detalhes que a aritmética. Falha se algum pixel for zero."""
    def geo_mean(window):
        return np.exp(np.mean(np.log(window + 1e-5)))
    return generic_filter(img.astype(np.float32), geo_mean, size=kernel_size).astype(np.uint8)

def filtro_media_harmonica(img, kernel_size=3):
    """Eficaz contra ruído sal (pixels brancos). Falha com pixels zero."""
    def harm_mean(window):
        return len(window) / np.sum(1.0 / (window + 1e-5))
    return generic_filter(img.astype(np.float32), harm_mean, size=kernel_size).astype(np.uint8)

def filtro_media_contraharmonica(img, kernel_size=3, Q=1.5):
    """Q > 0 elimina pimenta; Q < 0 elimina sal; Q = 0 equivale à aritmética."""
    def contra_harm(window):
        return np.sum(window**(Q+1)) / (np.sum(window**Q) + 1e-5)
    return generic_filter(img.astype(np.float32), contra_harm, size=kernel_size).astype(np.uint8)

In [ ]:
#  Comparação: filtros de média sobre ruído impulsivo (sal e pimenta)
img_ruido = add_impulse_noise(original)

resultados = [
    original, img_ruido,
    filtro_media_aritmetica(img_ruido),
    filtro_media_geometrica(img_ruido),
    filtro_media_harmonica(img_ruido),
    filtro_media_contraharmonica(img_ruido, Q=-1.5),  # Q < 0: elimina sal
]
titulos = [
    "Original", "Com Ruído Impulsivo",
    "Média Aritmética", "Média Geométrica",
    "Média Harmônica", "Contra-harmônica (Q=-1.5)",
]
mostrar_grade(resultados, titulos, figsize=(22, 4))

### 2.2 Filtros de Estatística de Ordem (não-lineares)

In [ ]:
# Filtros de ordem: mediana, máximo, mínimo, ponto médio 

def filtro_mediana(img, kernel_size=3):
    """Valor central da vizinhança ordenada. Melhor opção para ruído sal-e-pimenta."""
    return cv2.medianBlur(img, kernel_size)

def filtro_maximo(img, kernel_size=3):
    """Máximo local. Expande regiões claras; remove ruído pimenta (pixels pretos)."""
    return maximum_filter(img, size=kernel_size)

def filtro_minimo(img, kernel_size=3):
    """Mínimo local. Expande regiões escuras; remove ruído sal (pixels brancos)."""
    return minimum_filter(img, size=kernel_size)

def filtro_ponto_medio(img, kernel_size=3):
    """Média entre mínimo e máximo. Funciona bem para ruído uniforme e gaussiano."""
    min_img = minimum_filter(img, size=kernel_size)
    max_img = maximum_filter(img, size=kernel_size)
    return ((min_img.astype(np.uint16) + max_img.astype(np.uint16)) // 2).astype(np.uint8)

In [ ]:
# Comparação: filtros de ordem sobre ruído impulsivo 
img_ruido = add_impulse_noise(original)

resultados = [
    original, img_ruido,
    filtro_mediana(img_ruido),
    filtro_maximo(img_ruido),
    filtro_minimo(img_ruido),
    filtro_ponto_medio(img_ruido),
]
titulos = [
    "Original", "Com Ruído Impulsivo",
    "Mediana", "Máximo", "Mínimo", "Ponto Médio",
]
mostrar_grade(resultados, titulos, figsize=(22, 4))

### 2.3 Filtros do OpenCV (Blur, Mediana, Gaussiano)

In [ ]:
#  Comparação direta dos três filtros principais do OpenCV 
imagem_gray = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)

suav_media    = cv2.blur(imagem_gray, (15, 15))
suav_mediana  = cv2.medianBlur(imagem_gray, 15)
suav_gaussian = cv2.GaussianBlur(imagem_gray, (15, 15), 0)

mostrar_grade(
    [imagem_gray, suav_media, suav_mediana, suav_gaussian],
    ["Original", "Blur (Média) 15×15", "Mediana 15×15", "Gaussiano 15×15"],
    figsize=(20, 5)
)

print("""
Resumo de uso:
  Blur (Média)  → suavização rápida; borra bordas
  Mediana       → ideal para ruído sal-e-pimenta; preserva bordas
  Gaussiano     → padrão para pré-processar antes de detectar bordas
"""
)

### Faça Você mesmo — Aplicação de Filtros em Tomografia

**Objetivo:** aplicar filtros de suavização em uma tomografia com ruído (`imagens/tumografia-ruido.png`)
e comparar qual filtro produz o melhor resultado para imagens médicas.

**Tarefas:**
1. Carregue a imagem em escala de cinza
2. Aplique blur, gaussiano e mediana com kernel 5×5
3. Qual filtro preserva melhor as estruturas internas? Por quê?

---
## 3. Limiarização (Thresholding)

Técnica de **segmentar uma imagem em duas classes** com base na intensidade dos pixels:

> se f(x,y) > T → objeto (255);  caso contrário → fundo (0)

**A escolha de T é crítica** — um limiar mal calibrado perde detalhes ou inclui ruído.

> **Pré-requisito:** sempre converta para escala de cinza antes de limiarizar.

### 3.1 Tipos de Limiarização Simples

In [ ]:
# Carrega imagem de referência 
imagem = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)
T = 190  # limiar fixo para demonstração

tipos = {
    "Original":              imagem,
    "THRESH_BINARY":         cv2.threshold(imagem, T, 255, cv2.THRESH_BINARY)[1],
    "THRESH_BINARY_INV":     cv2.threshold(imagem, T, 255, cv2.THRESH_BINARY_INV)[1],
    "THRESH_TRUNC":          cv2.threshold(imagem, T, 255, cv2.THRESH_TRUNC)[1],
    "THRESH_TOZERO":         cv2.threshold(imagem, T, 255, cv2.THRESH_TOZERO)[1],
    "THRESH_TOZERO_INV":     cv2.threshold(imagem, T, 255, cv2.THRESH_TOZERO_INV)[1],
}

mostrar_grade(list(tipos.values()), list(tipos.keys()), figsize=(24, 4))

print(f"""
Legenda (T = {T}):
  BINARY     : x > T → 255,  caso contrário → 0
  BINARY_INV : x > T → 0,    caso contrário → 255
  TRUNC      : x > T → T,    caso contrário → x  (limita intensidade máxima)
  TOZERO     : x < T → 0,    caso contrário → x  (preserva valores altos)
  TOZERO_INV : x > T → 0,    caso contrário → x  (preserva valores baixos)
""")

#### Slider interativo — ajuste o limiar em tempo real

In [ ]:
from ipywidgets import interact, IntSlider

def update_threshold(threshold_value):
    img = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)
    _, limiarizada = cv2.threshold(img, threshold_value, 255, cv2.THRESH_BINARY)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(img, cmap='gray'); axes[0].set_title("Original"); axes[0].axis('off')
    axes[1].imshow(limiarizada, cmap='gray')
    axes[1].set_title(f"THRESH_BINARY  (T = {threshold_value})"); axes[1].axis('off')
    plt.tight_layout(); plt.show()

interact(update_threshold, threshold_value=IntSlider(min=0, max=255, step=1, value=128,
         description='Threshold:'));

### 3.2 Limiarização de Otsu

O método de Otsu (1979) **seleciona automaticamente T** maximizando a variância entre as duas classes.

**Quando funciona bem:** histogramas bimodais (dois picos separados).

**Quando falha:** iluminação desigual ou histogramas unimodais → use limiarização adaptativa.

In [ ]:
# Otsu: histograma e resultado 
imagem = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)
blur_otsu = cv2.GaussianBlur(imagem, (7, 7), 0)

ret, otsu_result = cv2.threshold(blur_otsu, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(imagem, cmap='gray'); axes[0].set_title("Original"); axes[0].axis('off')

axes[1].hist(blur_otsu.ravel(), 256, [0, 256], color='steelblue')
axes[1].axvline(ret, color='crimson', linewidth=2, label=f'T* Otsu = {int(ret)}')
axes[1].set_title("Histograma com T* (Otsu)"); axes[1].legend()

axes[2].imshow(otsu_result, cmap='gray')
axes[2].set_title(f"Resultado Otsu (T = {int(ret)})"); axes[2].axis('off')

plt.tight_layout(); plt.show()
print(f"Limiar ótimo encontrado por Otsu: {int(ret)}")

### 3.3 Limiarização Adaptativa

**Problema da limiarização global:** um único T falha quando a iluminação é desigual.

**Solução:** calcular um T local T(x,y) para cada pixel com base na sua vizinhança.

**Parâmetros:**
- `blockSize`: tamanho da vizinhança (sempre ímpar: 11, 51, 101)
- `C`: constante subtraída da média — controla a sensibilidade

**Dois métodos disponíveis:**
- `ADAPTIVE_THRESH_MEAN_C`: T(x,y) = média da vizinhança − C
- `ADAPTIVE_THRESH_GAUSSIAN_C`: T(x,y) = média gaussiana da vizinhança − C

In [ ]:
# Global vs Adaptativo 
imagem = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)
imagem_suav = cv2.GaussianBlur(imagem, (5, 5), 0)

_, global_thresh = cv2.threshold(imagem_suav, 180, 255, cv2.THRESH_BINARY)

adaptativa_mean = cv2.adaptiveThreshold(
    imagem_suav, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 195, 5)

adaptativa_gauss = cv2.adaptiveThreshold(
    imagem_suav, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 195, 5)

mostrar_grade(
    [imagem, global_thresh, adaptativa_mean, adaptativa_gauss],
    ["Original", "Global (T=180)", "Adaptativa Média", "Adaptativa Gaussiana"],
    figsize=(22, 5)
)

### Faça Você mesmo — Limiarização em Imagens Médicas

**Objetivo:** aplicar os três métodos de limiarização em imagens de lesões de pele (`imagens/skin/`)
e comparar qual segmenta melhor a região de interesse.

**Tarefas:**
1. Carregue cada imagem em escala de cinza
2. Aplique `GaussianBlur` antes de limiarizar
3. Compare Global, Otsu e Adaptativa — qual produz menos ruído no fundo?

In [ ]:
def processar_imagem_skin(caminho):
    img = # ler imagem em escala de cinza
    img = # suavizar com filtro gaussiano
    _, global_t    = # limiarização global com T=128
    limiar_adapt   = # limiarização adaptativa
    _, otsu_t      = # limiarização de Otsu
    return img, global_t, limiar_adapt, otsu_t

for caminho in glob("imagens/skin/*.*")[:2]:  # limita a 2 imagens para agilidade
    img, global_t, adapt_t, otsu_t = processar_imagem_skin(caminho)
    mostrar_grade(
        [img, global_t, adapt_t, otsu_t],
        ["Original", "Global (T=128)", "Adaptativa", "Otsu"],
        figsize=(18, 4)
    )

---
## 4. Operações Morfológicas

Operam sobre **imagens binárias** usando um **elemento estruturante** (kernel) como sonda.

| Operação | Definição | Efeito visual | Uso típico |
|----------|-----------|---------------|-----------|
| **Erosão** | mínimo local | encolhe regiões brancas | remove ruído, separa objetos |
| **Dilatação** | máximo local | expande regiões brancas | preenche buracos, conecta regiões |
| **Abertura** | erosão → dilatação | remove ruído sem alterar forma | limpar background ruidoso |
| **Fechamento** | dilatação → erosão | preenche buracos internos | fechar fissuras em objetos |

In [ ]:
# Erosão e Dilatação: exemplo com faixas 
imagem_rgb  = cv2.imread("imagens/044.png")
imagem_rgb  = cv2.cvtColor(imagem_rgb, cv2.COLOR_BGR2RGB)
imagem_gray = cv2.cvtColor(imagem_rgb, cv2.COLOR_RGB2GRAY)

# Pré-processamento
imagem_suav = cv2.GaussianBlur(imagem_gray, (5, 5), 0)
_, imagem_bin = cv2.threshold(imagem_suav, 180, 255, cv2.THRESH_BINARY_INV)

kernel = np.ones((5, 5), np.uint8)

dilatacao = cv2.dilate(imagem_bin, kernel, iterations=2)
erosao    = cv2.erode(imagem_bin,  kernel, iterations=1)
abertura  = cv2.morphologyEx(imagem_bin, cv2.MORPH_OPEN,  kernel)
fechamento = cv2.morphologyEx(imagem_bin, cv2.MORPH_CLOSE, kernel)

mostrar_grade(
    [imagem_bin, dilatacao, erosao, abertura, fechamento],
    ["Binarizada", "Dilatação (it=2)", "Erosão (it=1)", "Abertura", "Fechamento"],
    figsize=(24, 5)
)

### Exemplo prático: destacar estrelas em imagem espacial

In [ ]:
# Máscara por limiarização + dilatação 
imagem_space = cv2.imread("imagens/space.png")
imagem_space = cv2.cvtColor(imagem_space, cv2.COLOR_BGR2GRAY)

_, limiarizada = cv2.threshold(imagem_space, 150, 255, cv2.THRESH_BINARY)
kernel         = np.ones((5, 5), np.uint8)
dilatada       = cv2.dilate(limiarizada, kernel, iterations=1)
extraida       = cv2.bitwise_and(imagem_space, imagem_space, mask=dilatada)

mostrar_grade(
    [imagem_space, limiarizada, dilatada, extraida],
    ["Original (cinza)", "Limiarizada (T=150)", "Dilatação da máscara", "Região Extraída"],
    figsize=(22, 5)
)

### Faça Você mesmo — Morfologia em Impressão Digital

**Objetivo:** aplicar erosão e dilatação em `imagens/fingerprint.png` e observar como
diferentes elementos estruturantes afetam os traços da impressão.

**Tarefas:**
1. Binarize a impressão (inverta: cristas = branco)
2. Aplique erosão com `MORPH_RECT` e `MORPH_CROSS` (5×5)
3. Aplique abertura e fechamento — qual fecha as fissuras sem destruir as cristas?

In [ ]:
img_fp = # ler imagem em escala de cinza
_, img_fp = # limiarização global com T=128

elem_rect  = cv2.getStructuringElement(cv2.MORPH_RECT,  (5, 5))
elem_cross = cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))

erosao_rect    = # erosão com elemento retangular
erosao_cross   = # erosão com elemento cruz
abertura_rect  = # abertura com elemento retangular
fechamento_rect = # fechamento com elemento retangular

mostrar_grade(
    [img_fp, erosao_rect, erosao_cross, abertura_rect, fechamento_rect],
    ["Binarizada", "Erosão RECT", "Erosão CROSS", "Abertura RECT", "Fechamento RECT"],
    figsize=(24, 5)
)

---
## 5. Detecção de Bordas

Bordas são **transições abruptas de intensidade** — fronteiras entre regiões que formam
os contornos dos objetos.

Três detectores clássicos em ordem crescente de sofisticação:

| Detector | Base matemática | Saída | Limitação |
|----------|----------------|-------|-----------|
| **Sobel** | 1ª derivada (gradiente direcional) | Bordas espessas | Sensível a ruído |
| **Laplaciano** | 2ª derivada | Detecta qualquer direção | Amplifica ruído |
| **Canny** | Gaussiano + Sobel + NMS + Histerese | Bordas finas, contínuas | Mais lento |

> **Regra:** sempre suavize com Gaussiano antes de derivar — derivadas amplificam ruído.

### 5.1 Detector Sobel

Calcula derivadas parciais em X e Y usando os kernels de Sobel (3×3):

- **Gx** detecta bordas verticais (contraste horizontal)
- **Gy** detecta bordas horizontais (contraste vertical)
- **Magnitude:** |G| = √(Gx² + Gy²)
- **Direção:** θ = arctan(Gy / Gx)

In [ ]:
#  Sobel: gradientes direcionais 
imagem_gray = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)
imagem_suav = cv2.GaussianBlur(imagem_gray, (5, 5), 1.4)  # suaviza antes de derivar

# Gradiente em X e Y
sobel_x = cv2.Sobel(imagem_suav, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(imagem_suav, cv2.CV_64F, 0, 1, ksize=3)

# Magnitude combinada
sobel_mag = cv2.magnitude(sobel_x, sobel_y)
sobel_mag = np.clip(sobel_mag, 0, 255).astype(np.uint8)

# Converter para exibição
sobel_x_abs = cv2.convertScaleAbs(sobel_x)
sobel_y_abs = cv2.convertScaleAbs(sobel_y)

mostrar_grade(
    [imagem_gray, sobel_x_abs, sobel_y_abs, sobel_mag],
    ["Original", "Sobel Gx (bordas verticais)", "Sobel Gy (bordas horizontais)", "Sobel Magnitude |G|"],
    figsize=(22, 5)
)

print("""
Interpretação:
  Gx alto → variação horizontal de intensidade (bordas verticais)
  Gy alto → variação vertical de intensidade   (bordas horizontais)
  |G| alto → borda forte em qualquer direção
"""
)

### 5.2 Detector Laplaciano

O Laplaciano usa a **segunda derivada** — detecta bordas em qualquer direção
calculando a divergência do gradiente:

> ∇²f = ∂²f/∂x² + ∂²f/∂y²

**Vantagem:** isotrópico (não depende da orientação da borda).
**Desvantagem:** muito sensível a ruído — obrigatório suavizar antes.

In [ ]:
#  Laplaciano 
imagem_suav = cv2.GaussianBlur(imagem_gray, (5, 5), 1.4)

laplaciano   = cv2.Laplacian(imagem_suav, cv2.CV_64F, ksize=3)
laplaciano_abs = cv2.convertScaleAbs(laplaciano)

# Laplaciano of Gaussian (LoG) — versão mais robusta
log_blur  = cv2.GaussianBlur(imagem_gray, (9, 9), 2.0)
log_result = cv2.Laplacian(log_blur, cv2.CV_64F, ksize=3)
log_abs    = cv2.convertScaleAbs(log_result)

mostrar_grade(
    [imagem_gray, imagem_suav, laplaciano_abs, log_abs],
    ["Original", "Gaussiano pré-aplicado", "Laplaciano", "Laplaciano of Gaussian (LoG)"],
    figsize=(22, 5)
)

print("""
Nota: pixels em zero-crossing no Laplaciano indicam a localização exata da borda.
LoG suaviza primeiro, tornando a detecção mais robusta a ruído.
"""
)

### 5.3 Detector Canny — padrão da indústria

O algoritmo de Canny (1986, MIT) opera em **4 estágios:**

1. **Filtro Gaussiano:** remove ruído de alta frequência sem destruir bordas reais
2. **Busca de Gradientes:** calcula magnitude e direção via Sobel (X e Y)
3. **Supressão Não-Máxima (NMS):** afina cada borda para exatamente 1 pixel de largura
4. **Histerese:** dois limiares (T_alto, T_baixo); bordas fracas só sobrevivem se conectadas a bordas fortes

**Parâmetros-chave:** `threshold1` (T_baixo) e `threshold2` (T_alto). Proporção recomendada: 1:2 ou 1:3.

In [ ]:
#  Canny com diferentes limiares 
imagem_gray = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)

canny_sensivel  = cv2.Canny(imagem_gray, 20,  60)   # detecta mais bordas (inclusive ruído)
canny_balanceado = cv2.Canny(imagem_gray, 40,  80)  # equilíbrio recomendado
canny_restrito  = cv2.Canny(imagem_gray, 80, 160)   # apenas bordas muito fortes

mostrar_grade(
    [imagem_gray, canny_sensivel, canny_balanceado, canny_restrito],
    ["Original", "Canny (T=20/60) sensível", "Canny (T=40/80) padrão", "Canny (T=80/160) restrito"],
    figsize=(22, 5)
)

### 5.4 Comparação: Sobel × Laplaciano × Canny

In [ ]:
# Comparação lado a lado dos três detectores 
imagem_suav = cv2.GaussianBlur(imagem_gray, (5, 5), 1.4)

sobel_final  = cv2.convertScaleAbs(cv2.magnitude(
    cv2.Sobel(imagem_suav, cv2.CV_64F, 1, 0, ksize=3),
    cv2.Sobel(imagem_suav, cv2.CV_64F, 0, 1, ksize=3)
))
lap_final    = cv2.convertScaleAbs(cv2.Laplacian(imagem_suav, cv2.CV_64F, ksize=3))
canny_final  = cv2.Canny(imagem_gray, 40, 80)

mostrar_grade(
    [imagem_gray, sobel_final, lap_final, canny_final],
    ["Original", "Sobel |G|", "Laplaciano", "Canny (padrão)"],
    figsize=(22, 5)
)

print("""
Quando usar cada detector:
  Sobel     → análise direcional rápida; tempo real/baixo custo
  Laplaciano → detecção isotrópica; combine com LoG para mais robustez
  Canny     → padrão para OCR, segmentação, navegação autônoma
"""
)

---
## 6. Segmentação de Imagens

**Objetivo:** separar a imagem em **regiões significativas** — cada região representa
um objeto ou área distinta.

Pipeline típico:
```
Imagem colorida → Binarizar → Morfologia → Contornos → Análise
                 (threshold)  (erode/dilate) (findContours) (área, bbox, hull)
```
> **Regra de bolso:** se o resultado final está ruim, volte uma etapa — o problema
quase sempre é binarização instável ou morfologia mal dimensionada.

### 6.1 Crescimento de Região

Agrupa pixels vizinhos que compartilham um critério de similaridade (intensidade, cor ou textura),
partindo de uma **semente** inicial.

**Algoritmo (4 passos):**
1. Escolher semente (pixel inicial)
2. Definir critério: |I(p) − I(semente)| < T
3. Expandir para vizinhos 4 ou 8-conectados que satisfaçam o critério
4. Parar quando não houver vizinhos elegíveis

**Cuidado:** T pequeno → regiões fragmentadas; T grande → vazamento entre objetos.

In [ ]:
# Crescimento de região: implementação BFS 

def region_growing(image, seed, threshold):
    """Segmenta uma região a partir de uma semente usando BFS."""
    h, w = image.shape
    mask    = np.zeros_like(image, dtype=np.uint8)
    visited = np.zeros_like(image, dtype=bool)
    seed_val = int(image[seed[0], seed[1]])

    queue = [seed]
    visited[seed[0], seed[1]] = True
    neighbors = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

    while queue:
        y, x = queue.pop(0)
        mask[y, x] = 255
        for dy, dx in neighbors:
            ny, nx = y + dy, x + dx
            if 0 <= ny < h and 0 <= nx < w:
                if not visited[ny, nx] and abs(int(image[ny, nx]) - seed_val) <= threshold:
                    visited[ny, nx] = True
                    queue.append((ny, nx))
    return mask

def full_region_growing(image, threshold):
    """Rotula todas as regiões da imagem por crescimento de região."""
    h, w    = image.shape
    visited = np.zeros_like(image, dtype=bool)
    labels  = np.zeros_like(image, dtype=np.int32)
    label   = 1
    neighbors = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

    for y in range(h):
        for x in range(w):
            if not visited[y, x]:
                seed_val = int(image[y, x])
                queue    = [(y, x)]
                visited[y, x] = True
                while queue:
                    cy, cx = queue.pop(0)
                    labels[cy, cx] = label
                    for dy, dx in neighbors:
                        ny, nx = cy + dy, cx + dx
                        if 0 <= ny < h and 0 <= nx < w:
                            if not visited[ny, nx] and abs(int(image[ny, nx]) - seed_val) <= threshold:
                                visited[ny, nx] = True
                                queue.append((ny, nx))
                label += 1
    return labels

In [ ]:
# Demonstração: crescimento de região completo 
img_orig = cv2.imread("imagens/044.png", cv2.IMREAD_GRAYSCALE)

# Pré-processamento para segmentação
blur_rg  = cv2.GaussianBlur(img_orig, (5, 5), 0)
_, img_bin_rg = cv2.threshold(blur_rg, 180, 255, cv2.THRESH_BINARY_INV)
img_eroded = cv2.erode(img_bin_rg, np.ones((5,5), np.uint8), iterations=1)

# Crescimento de região
labels = full_region_growing(img_eroded, threshold=15)

# Normaliza para exibição colorida
label_norm = (labels.astype(np.float32) / (labels.max() + 1) * 255).astype(np.uint8)
label_colorido = cv2.applyColorMap(label_norm, cv2.COLORMAP_RAINBOW)
label_colorido = cv2.cvtColor(label_colorido, cv2.COLOR_BGR2RGB)

mostrar_grade(
    [img_orig, img_eroded, label_colorido],
    ["Original", "Pré-processada", f"Crescimento de Região ({labels.max()} regiões)"],
    cmap=None, figsize=(20, 6)
)

### 6.2 Componentes Conectados

Recebe uma imagem binária e rotula cada **blob** de pixels foreground com um ID inteiro distinto.

**Diferença para crescimento de região:** aqui não há semente nem critério de similaridade —
todo pixel foreground entra em algum componente, sempre.

`cv2.connectedComponentsWithStats` retorna para cada componente:
- `labels`: mapa de rótulos inteiros
- `stats`: [x, y, largura, altura, área] de cada componente
- `centroids`: centroide (cx, cy) de cada componente

In [ ]:
# Componentes Conectados: exemplo com parafusos e porcas 
fn_parafusos = glob(os.path.join('imagens', 'Parafusos', '*.jpg'))

if fn_parafusos:
    img_orig_cc = cv2.imread(fn_parafusos[0], 1)
    B, G, R     = cv2.split(img_orig_cc)

    # Pré-processamento: filtro bilateral + blur + binarização + dilatação
    img_prep = cv2.bilateralFilter(G, 1, 90, 90)
    img_prep = cv2.blur(img_prep, (5, 5))
    img_prep = cv2.threshold(img_prep, 190, 255, cv2.THRESH_BINARY)[1]
    img_prep = cv2.dilate(img_prep, np.ones((4,4), np.uint8), iterations=1)

    # Componentes conectados
    num_labels, labels_cc, stats, centroids = cv2.connectedComponentsWithStats(
        img_prep, 4, cv2.CV_8U)

    # Colorir cada componente com uma cor diferente
    label_hue  = np.uint8(179 * labels_cc / np.max(labels_cc))
    blank_ch   = 255 * np.ones_like(label_hue)
    labeled_img = cv2.merge([label_hue, blank_ch, blank_ch])
    labeled_img = cv2.cvtColor(labeled_img, cv2.COLOR_HSV2BGR)
    labeled_img[labels_cc == 0] = 0  # fundo = preto

    # Análise por área
    df_areas = pd.DataFrame(stats, columns=['X','Y','W','H','AREA'])
    df_areas.drop(df_areas.index[0], inplace=True)  # remove fundo
    parafusos = df_areas[df_areas['AREA'] > 900]
    porcas    = df_areas[df_areas['AREA'] <= 900]

    print(f"Componentes detectados (sem fundo): {len(df_areas)}")
    print(f"  Parafusos (AREA > 900): {len(parafusos)}")
    print(f"  Porcas    (AREA ≤ 900): {len(porcas)}")
    print(df_areas.to_string())

    img_orig_rgb = cv2.cvtColor(img_orig_cc, cv2.COLOR_BGR2RGB)
    labeled_rgb  = cv2.cvtColor(labeled_img, cv2.COLOR_BGR2RGB)
    mostrar_grade(
        [img_orig_rgb, img_prep, labeled_rgb],
        ["Original", "Binarizada + Dilatada", f"Componentes Conectados ({num_labels-1} objetos)"],
        cmap=None, figsize=(20, 6)
    )
else:
    print("Pasta imagens/Parafusos/ não encontrada. Demonstrando com imagem sintética.")

    # Demonstração sintética
    sintetica = np.zeros((200, 600), dtype=np.uint8)
    cv2.circle(sintetica,    (100, 100), 60, 255, -1)
    cv2.rectangle(sintetica, (250, 40), (380, 160), 255, -1)
    cv2.circle(sintetica,    (500, 100), 40, 255, -1)
    cv2.rectangle(sintetica, (150, 130), (200, 180), 255, -1)

    num_labels, labels_sint, stats_sint, cents = cv2.connectedComponentsWithStats(sintetica, 8, cv2.CV_8U)

    label_hue2   = np.uint8(179 * labels_sint / max(np.max(labels_sint), 1))
    blank2       = 255 * np.ones_like(label_hue2)
    labeled_sint = cv2.cvtColor(cv2.merge([label_hue2, blank2, blank2]), cv2.COLOR_HSV2BGR)
    labeled_sint[labels_sint == 0] = 0
    labeled_sint_rgb = cv2.cvtColor(labeled_sint, cv2.COLOR_BGR2RGB)

    # Anotar centroide de cada componente
    anotada = labeled_sint_rgb.copy()
    for i, (cx, cy) in enumerate(cents[1:], 1):
        cv2.circle(anotada, (int(cx), int(cy)), 5, (255,255,255), -1)
        cv2.putText(anotada, f"ID {i} | area={stats_sint[i,4]}",
                    (int(cx)-30, int(cy)-12), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)

    mostrar_grade(
        [sintetica, labeled_sint_rgb, anotada],
        ["Sintética (binária)", f"{num_labels-1} componentes coloridos", "Com centroides e áreas"],
        cmap=None, figsize=(20, 6)
    )

### 6.3 Segmentação por Contornos

Um **contorno** é uma curva fechada ao redor de um objeto — essencial para detecção,
classificação de formas e medição de áreas.

**Parâmetros de `findContours`:**

| Parâmetro | Opção | Significado |
|-----------|-------|-------------|
| `mode` | `RETR_EXTERNAL` | Apenas contornos externos (ignora buracos internos) |
| `mode` | `RETR_LIST` | Todos os contornos sem hierarquia |
| `method` | `CHAIN_APPROX_SIMPLE` | Comprime segmentos retos; salva só os vértices |
| `method` | `CHAIN_APPROX_NONE` | Todos os pontos da borda (usa muito mais memória) |

In [ ]:
#  Extração e desenho de contornos 
imagem_obj     = cv2.imread("imagens/044.png")
imagem_obj_rgb = cv2.cvtColor(imagem_obj, cv2.COLOR_BGR2RGB)
imagem_obj_gray = cv2.cvtColor(imagem_obj, cv2.COLOR_BGR2GRAY)

# Detecção de bordas como entrada para findContours
bordas = cv2.Canny(imagem_obj_gray, 100, 110)

# Contornos externos (RETR_EXTERNAL) vs todos (RETR_LIST)
contornos_ext, _ = cv2.findContours(bordas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos_all, _ = cv2.findContours(bordas, cv2.RETR_LIST,     cv2.CHAIN_APPROX_SIMPLE)

img_ext = imagem_obj_rgb.copy()
img_all = imagem_obj_rgb.copy()
cv2.drawContours(img_ext, contornos_ext, -1, (0, 255, 0), 2)
cv2.drawContours(img_all, contornos_all, -1, (255, 0, 0), 2)

print(f"RETR_EXTERNAL: {len(contornos_ext)} contornos")
print(f"RETR_LIST:     {len(contornos_all)} contornos (inclui internos)")

mostrar_grade(
    [imagem_obj_rgb, bordas, img_ext, img_all],
    ["Original", "Bordas (Canny)", f"Externos ({len(contornos_ext)})", f"Todos ({len(contornos_all)})"],
    cmap=None, figsize=(22, 5)
)

### 6.4 Análise e Ordenação de Contornos

In [ ]:
#  Funções de análise de contornos 

def areas_contornos(contornos):
    return [cv2.contourArea(c) for c in contornos]

def label_centroide(imagem, contorno, identificacao, cor=(0, 255, 0)):
    """Calcula centroide via momentos e escreve um label na imagem."""
    M = cv2.moments(contorno)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        cv2.putText(imagem, str(identificacao), (cx, cy),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, cor, 2)

def posicao_x(contorno):
    """Retorna a coordenada X do centroide para ordenação esquerda → direita."""
    M = cv2.moments(contorno)
    return int(M["m10"] / M["m00"]) if M["m00"] != 0 else 0

In [ ]:
#  Pipeline completo: filtrar, ordenar e rotular contornos 
imagem = cv2.imread("imagens/044.png")
imagem = cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB)
imagem_gray_c  = cv2.cvtColor(imagem, cv2.COLOR_RGB2GRAY)
imagem_gray_suav = cv2.GaussianBlur(imagem_gray_c, (3,3), 0)
bordas_c = cv2.Canny(imagem_gray_suav, 40, 110)

contornos_c, _ = cv2.findContours(bordas_c, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Filtrar contornos pequenos (ruído)
contornos_filtrados = [c for c in contornos_c if cv2.contourArea(c) > 30]
print(f"Total: {len(contornos_c)} | Após filtro (area > 30): {len(contornos_filtrados)}")

# Ordenação por área (decrescente) e por posição X (esquerda → direita)
contornos_area = sorted(contornos_filtrados, key=cv2.contourArea, reverse=True)
contornos_x    = sorted(contornos_filtrados, key=posicao_x,       reverse=False)

img_por_area = imagem.copy()
img_por_x    = imagem.copy()
for idx, c in enumerate(contornos_area): label_centroide(img_por_area, c, idx+1, (255,0,0))
for idx, c in enumerate(contornos_x):    label_centroide(img_por_x,    c, idx+1, (0,200,0))

mostrar_grade(
    [img_por_area, img_por_x],
    ["Ordenado por Área (vermelho)", "Ordenado por Posição X (verde)"],
    cmap=None, figsize=(18, 7)
)

### 6.5 Casca Convexa (Convex Hull)

A **casca convexa** é o menor polígono convexo que envolve todos os pontos de um contorno.

**Diferença em relação ao bounding rect:**
- `boundingRect` → retângulo alinhado aos eixos (mais simples, ignora forma)
- `convexHull`   → polígono convexo ajustado à silhueta do objeto (mais preciso)

**Aplicações:** detecção de gestos de mão (contagem de dedos via defeitos convexos),
inspeção industrial de formas irregulares, análise de silhuetas.

In [ ]:
#  Casca Convexa: comparação com bounding rect 
imagem_cv  = cv2.imread("imagens/044.png")
imagem_cv_rgb  = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2RGB)
imagem_cv_gray = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)

bordas_cv = cv2.Canny(imagem_cv_gray, 1, 100)
contornos_cv, _ = cv2.findContours(bordas_cv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

img_bbox = imagem_cv_rgb.copy()
img_hull = imagem_cv_rgb.copy()

for contorno in contornos_cv:
    # Bounding Rect (retângulo alinhado)
    x, y, w, h = cv2.boundingRect(contorno)
    cv2.rectangle(img_bbox, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Casca Convexa
    hull = cv2.convexHull(contorno)
    cv2.drawContours(img_hull, [hull], 0, (255, 100, 0), 2)

mostrar_grade(
    [imagem_cv_rgb, img_bbox, img_hull],
    ["Original", "Bounding Rect (verde)", "Casca Convexa (laranja)"],
    cmap=None, figsize=(20, 6)
)

In [ ]:
# Casca Convexa com defeitos convexos: exemplo em forma de mão sintética 
# Cria uma forma côncava sintética (simulando uma silhueta com reentrâncias)
sintetica_hand = np.zeros((300, 400), dtype=np.uint8)

# Pontos que simulam dedos (forma de "mão")
pts = np.array([
    [150, 280], [120, 180], [100, 120], [110, 60], [130, 100],
    [150, 130], [160, 80],  [175, 50],  [190, 80], [200, 130],
    [210, 90],  [230, 60],  [240, 90],  [245, 130],[255, 100],
    [270, 70],  [280, 100], [275, 150], [260, 200],[230, 250],
    [200, 280]
], np.int32)
cv2.fillPoly(sintetica_hand, [pts], 255)

# Contorno e hull
contorno_hand, _ = cv2.findContours(sintetica_hand, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
hull_hand = cv2.convexHull(contorno_hand[0])

# Defeitos convexos: regiões côncavas entre dedos
hull_idx = cv2.convexHull(contorno_hand[0], returnPoints=False)
defeitos  = cv2.convexityDefects(contorno_hand[0], hull_idx)

img_hand_color = cv2.cvtColor(sintetica_hand, cv2.COLOR_GRAY2RGB)
cv2.drawContours(img_hand_color, contorno_hand, -1, (0, 200, 0), 2)
cv2.drawContours(img_hand_color, [hull_hand],    0, (255, 100, 0), 2)

n_defeitos = 0
if defeitos is not None:
    for d in defeitos:
        s, e, f, dist = d[0]
        if dist > 5000:  # filtra pequenos defeitos
            far = tuple(contorno_hand[0][f][0])
            cv2.circle(img_hand_color, far, 6, (255, 0, 0), -1)
            n_defeitos += 1

mostrar_grade(
    [sintetica_hand, img_hand_color],
    ["Forma sintética", f"Contorno (verde) + Hull (laranja) + Defeitos convexos ({n_defeitos} pontos vermelhos)"],
    cmap=None, figsize=(14, 6)
)
print(f"Defeitos convexos detectados (possíveis 'vales' entre dedos): {n_defeitos}")

---
## 7. Case Prático — Manchas de Óleo no Nordeste

Segmentação por contornos aplicada ao mapeamento de manchas de óleo no litoral do nordeste
brasileiro, auxiliando equipes de resgate ambiental na priorização das áreas afetadas.

**Pipeline aplicado:**
```
Imagem colorida → Grayscale → GaussianBlur → Limiarização → Canny → findContours → Filtro de área
```

In [ ]:
#  Case: manchas de óleo 
imagem_oleo     = cv2.imread("imagens/oleo_no_nordeste.jpeg")
imagem_oleo_rgb = cv2.cvtColor(imagem_oleo, cv2.COLOR_BGR2RGB)
imagem_oleo_gray = cv2.cvtColor(imagem_oleo, cv2.COLOR_BGR2GRAY)

# Etapa 1: sem pré-processamento — muito ruído
bordas_sem_prep = cv2.Canny(imagem_oleo_gray, 1, 100)

# Etapa 2: Gaussiano + limiarização
suav_oleo = cv2.GaussianBlur(imagem_oleo_gray, (19, 19), 0)
_, limiar_oleo = cv2.threshold(suav_oleo, 95, 255, cv2.THRESH_BINARY)
bordas_com_prep = cv2.Canny(limiar_oleo, 1, 100)

# Etapa 3: encontrar e filtrar contornos por área
contornos_oleo, _ = cv2.findContours(bordas_com_prep, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contornos_filtrados_oleo = [c for c in contornos_oleo if cv2.contourArea(c) > 10000]

img_resultado_oleo = imagem_oleo_rgb.copy()
cv2.drawContours(img_resultado_oleo, contornos_filtrados_oleo, -1, (0, 255, 0), 2)

print(f"Contornos totais:   {len(contornos_oleo)}")
print(f"Manchas relevantes (area > 10.000 px²): {len(contornos_filtrados_oleo)}")

mostrar_grade(
    [imagem_oleo_rgb, bordas_sem_prep, bordas_com_prep, img_resultado_oleo],
    ["Original", "Canny sem pré-proc. (ruidoso)", "Canny com pré-proc.", "Manchas segmentadas"],
    cmap=None, figsize=(24, 6)
)

---
## 9. Bônus — Esteganografia com LSB

# O que é Esteganografia?

Esteganografia é a prática de ocultar informações dentro de outros dados de forma que a presença da mensagem escondida seja imperceptível para observadores não autorizados. A palavra tem origem no grego, significando literalmente "escrita escondida". Diferentemente da criptografia, que torna a mensagem ilegível para quem não possui a chave de decodificação, a esteganografia foca em ocultar a existência da comunicação.

A informação escondida pode ser um texto, uma imagem, um áudio, um vídeo ou outro tipo de dado, e o "container" pode ser, por exemplo, uma imagem, um arquivo de áudio ou um vídeo.

## Como funciona?

A esteganografia utiliza técnicas para inserir dados ocultos dentro de arquivos digitais sem comprometer significativamente sua aparência ou funcionalidade. Um exemplo comum é a manipulação dos **bits menos significativos (LSB - Least Significant Bits)** de uma imagem ou áudio, que permite alterar os dados sem modificar perceptivelmente o arquivo original.

### Exemplo básico: LSB em imagens
Em uma imagem digital, os valores de cor de cada pixel são representados por números binários. Alterar os bits menos significativos (como o último bit) não altera a cor perceptível ao olho humano, mas permite armazenar informações ocultas.

<p align="center">
  <img src="https://miro.medium.com/v2/resize:fit:4800/format:webp/1*d4r7n75kNGyVeXsJKDpI0A.png" alt="Exemplo de Esteganografia - LSB em imagens">
</p>


### Outras Formas de Esteganografia

Além do método de **bits menos significativos (LSB)**, existem diversas outras formas de esteganografia que podem ser usadas dependendo do tipo de dado e do nível de segurança desejado. Aqui estão três exemplos:

#### 1. **Esteganografia em Domínio da Transformada**
Neste método, as informações são ocultadas no domínio da frequência em vez do domínio espacial. Um exemplo comum é a **Transformada Discreta de Fourier (DFT)** ou **Transformada Discreta de Coseno (DCT)**, amplamente utilizadas em imagens JPEG. As informações escondidas são inseridas nos coeficientes de frequência da imagem, tornando-as mais resistentes a compressões e modificações básicas.

- **Aplicação prática:** Ocultação de mensagens em imagens JPEG que passam por compressão sem perder os dados escondidos.

#### 2. **Esteganografia em Arquivos de Áudio**
Neste método, informações são escondidas em arquivos de áudio por meio de técnicas como:
- Manipulação de **frequências inaudíveis** (ex.: sons em frequências que o ouvido humano não consegue perceber).
- Modificação de **amplitudes** ou **fases** sem impactar a qualidade auditiva.

- **Aplicação prática:** Mensagens secretas enviadas dentro de músicas ou gravações de voz.

#### 3. **Esteganografia Baseada em Vídeo**
Arquivos de vídeo são um excelente meio para esteganografia, já que possuem uma grande quantidade de frames e canais de dados, permitindo:
- Inserir informações em **frames individuais** ou em metadados do vídeo.
- Modificar pequenas partes da cor ou do áudio sincronizado sem alterar a percepção do vídeo.

- **Aplicação prática:** Comunicação secreta através de vídeos enviados por plataformas públicas, como redes sociais.

Essas técnicas oferecem alternativas ao LSB, cada uma com vantagens específicas para diferentes contextos e tipos de mídia, dependendo do nível de complexidade e da resistência necessária contra detecção ou alterações no arquivo.



---

## Aplicações Reais da Esteganografia

### 1. **Segurança da Informação**
- **Comunicação Segura:** Mensagens escondidas podem ser transmitidas dentro de imagens ou arquivos de áudio para evitar a detecção em redes monitoradas.
- **Proteção de Propriedade Intelectual:** Marcas d'água invisíveis podem ser usadas para proteger direitos autorais de imagens, vídeos e áudios.

### 2. **Inteligência e Espionagem**
- **Transmissão Secreta de Dados:** Agentes podem usar esteganografia para esconder mensagens dentro de mídias digitais, como vídeos ou fotografias, dificultando a detecção por partes adversárias.
- **Codificação de Documentos Sensíveis:** Informações confidenciais podem ser ocultadas em arquivos comuns para evitar sua identificação por sistemas de monitoramento.

### 3. **Autenticação de Dados**
- **Marcas d’água digitais:** Aplicada em documentos digitais e conteúdo multimídia para garantir autenticidade e prevenir fraudes. Bancos e empresas utilizam esse método para verificar a origem e integridade de arquivos.

### 4. **Análise Forense**
- Investigadores podem utilizar esteganografia para recuperar mensagens escondidas em arquivos digitais, geralmente em casos relacionados a crimes cibernéticos.

### 5. **Armazenamento Encoberto**
- Arquivos sigilosos podem ser armazenados em mídias comuns (como imagens ou músicas) para evitar atenção indesejada em dispositivos confiscados.

---

## Exemplos de Casos de Uso

- **Mensagens em redes sociais:** Imagens publicadas podem conter dados escondidos que só podem ser extraídos por destinatários com conhecimento técnico.
- **Compartilhamento de chaves criptográficas:** Esteganografia pode ser usada para compartilhar chaves de criptografia de forma discreta, evitando a detecção por sistemas de vigilância.
- **Cibersegurança empresarial:** Empresas podem marcar digitalmente seus documentos para verificar vazamentos e rastrear a fonte.


---

## Esteganografia: Uso de uma Imagem para esconder uma Frase

O exemplo abaixo demostra como podemos usar uma imagem para escoder uma frase usando a estratégia de LSB.

A técnica LSB (Least Significant Bit) altera o bit menos significativo
de cada componente de cor (R, G, B) da imagem para armazenar os bits
da mensagem. Essa alteração é imperceptível ao olho humano, pois
afeta de maneira mínima o pixel original.

In [ ]:
from PIL import Image

def encoder(imagem_entrada, mensagem, imagem_saida):
    """
    Esconde uma mensagem em uma imagem utilizando o método LSB (Least Significant Bit).

    Parâmetros:
    - imagem_entrada: Caminho para a imagem onde a mensagem será escondida.
    - mensagem: A mensagem a ser escondida (texto simples).
    - imagem_saida: Caminho para salvar a imagem com a mensagem escondida.
    """
    # 1. Abrir e preparar a imagem para manipulação
    print(f"Abrindo a imagem '{imagem_entrada}' para codificação...")
    imagem = Image.open(imagem_entrada)
    imagem = imagem.convert("RGB")  # Converter a imagem para formato RGB
    pixels = imagem.load()  # Obter os pixels da imagem para manipulação

    # 2. Converter a mensagem em binário e adicionar um marcador de fim
    print("Convertendo a mensagem para formato binário...")
    mensagem_binaria = ''.join(format(ord(char), '08b') for char in mensagem) + '00000000'
    print(f"Mensagem binária: {mensagem_binaria[:64]}... (truncado para visualização)")

    # 3. Verificar se a mensagem cabe na imagem
    largura, altura = imagem.size
    total_pixels = largura * altura
    print(f"Resolução da imagem: {largura}x{altura} (total de {total_pixels} pixels).")
    if len(mensagem_binaria) > total_pixels * 3:
        raise ValueError("A mensagem é muito longa para ser escondida nesta imagem.")
    print("Mensagem confirmada como cabendo na imagem.")

    # 4. Inserir os bits da mensagem nos pixels da imagem
    print("Inserindo os bits da mensagem nos pixels da imagem...")
    mensagem_index = 0  # Índice para rastrear o progresso na mensagem
    for y in range(altura):
        for x in range(largura):
            if mensagem_index >= len(mensagem_binaria):  # Se toda a mensagem foi processada, sair do loop
                break

            # Obter os valores de cor do pixel atual
            r, g, b = pixels[x, y]

            # Alterar o LSB de cada canal de cor, se ainda houver bits para esconder
            if mensagem_index < len(mensagem_binaria):
                r = (r & ~1) | int(mensagem_binaria[mensagem_index])  # Modificar o LSB do vermelho (R)
                mensagem_index += 1
            if mensagem_index < len(mensagem_binaria):
                g = (g & ~1) | int(mensagem_binaria[mensagem_index])  # Modificar o LSB do verde (G)
                mensagem_index += 1
            if mensagem_index < len(mensagem_binaria):
                b = (b & ~1) | int(mensagem_binaria[mensagem_index])  # Modificar o LSB do azul (B)
                mensagem_index += 1

            # Atualizar o pixel na imagem
            pixels[x, y] = (r, g, b)

    # 5. Salvar a nova imagem com a mensagem escondida
    print(f"Salvando a imagem com a mensagem escondida em '{imagem_saida}'...")
    imagem.save(imagem_saida)
    print("Mensagem escondida com sucesso!")


In [ ]:
def decoder(imagem_entrada):
    """
    Revela uma mensagem escondida em uma imagem utilizando o método LSB (Least Significant Bit).

    Parâmetros:
    - imagem_entrada: Caminho para a imagem com a mensagem escondida.

    Retorna:
    - A mensagem revelada.
    """
    # Abrir a imagem
    imagem = Image.open(imagem_entrada)
    imagem = imagem.convert("RGB")
    pixels = imagem.load()

    mensagem_binaria = ""
    largura, altura = imagem.size

    # Iterar pelos pixels da imagem para extrair os bits da mensagem
    for y in range(altura):
        for x in range(largura):
            r, g, b = pixels[x, y]
            mensagem_binaria += str(r & 1)  # Obter LSB de R
            mensagem_binaria += str(g & 1)  # Obter LSB de G
            mensagem_binaria += str(b & 1)  # Obter LSB de B

    # Converter os bits binários de volta para texto
    mensagem = ""
    for i in range(0, len(mensagem_binaria), 8):
        byte = mensagem_binaria[i:i+8]
        if byte == "00000000":  # Verificar o marcador de fim de mensagem
            break
        mensagem += chr(int(byte, 2))

    return mensagem

In [ ]:
# Caminhos das imagens
imagem_original = "imagens/lena.jpg"
imagem_com_mensagem = "imagens/lena_esteganografia.png"

# Mensagem a ser escondida
mensagem_secreta = "Aula Visao Comp 250326"

# Esconder a mensagem
encoder(imagem_original, mensagem_secreta, imagem_com_mensagem)

# Revelar a mensagem
mensagem_revelada = decoder(imagem_com_mensagem)
print("Mensagem revelada:", mensagem_revelada)

In [ ]:
!pip install qrcode pyzbar

In [ ]:
!apt-get update

In [ ]:
!apt install libzbar0

In [ ]:
import qrcode
import numpy as np
from PIL import Image
from pyzbar.pyzbar import decode
import matplotlib.pyplot as plt
import os

class QrCode:
    def __init__(self, message):
        print("Gerando o QR Code com a mensagem fornecida...")
        self.img_qrcode = qrcode.make(message)
        self.img_encoded = None

    def encode(self, img_real):
        print(f"Codificando o QR Code dentro da imagem '{img_real}'...")
        img_oficial = Image.open(img_real).convert('L')
        secret = self.img_qrcode.resize(img_oficial.size, resample=Image.NEAREST).convert('L')

        data_c = np.array(img_oficial, dtype=np.int16)
        data_s = np.array(secret, dtype=np.int16)
        data_s_bin = (data_s > 127).astype(np.int16)

        # Injeção no bit menos significativo (LSB)
        res = (data_c & ~1) | (data_s_bin & 1)

        # Conversão de volta para PIL Image e salvamento
        self.img_encoded = Image.fromarray(res.astype(np.uint8), mode='L')

        output_path = "imagens/Lena_qrcode.png"
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        self.img_encoded.save(output_path)

        print(f"QR Code codificado e salvo com sucesso em {output_path}!")

    def decode(self):
        if self.img_encoded is None:
            raise RuntimeError("Imagem codificada não encontrada.")
        data_s = np.array(self.img_encoded, dtype=np.uint8) & 1
        intermediate_result = (data_s * np.uint8(255)).astype(np.uint8)
        return Image.fromarray(intermediate_result).convert("L")

    def get_text(self):
        qr_image = self.decode()
        decoded_data = decode(qr_image)
        if decoded_data:
            text = decoded_data[0].data.decode('utf-8')
            return text
        return None

    def plot_bitplanes(self):
        data = np.array(self.img_encoded, dtype=np.uint8)
        out = []
        for k in range(7, -1, -1):
            res = ((data >> k) & 1) * 255
            out.append(res.astype(np.uint8))
        fig, axes = plt.subplots(1, len(out), figsize=(15, 5))
        for i, plane in enumerate(out):
            axes[i].imshow(plane, cmap="gray")
            axes[i].axis("off")
            axes[i].set_title(f"BP {7-i}")
        plt.show()

# Testando a correção
mensagem = "Visao COmp é isso ai"
qrcode_obj = QrCode(mensagem)
qrcode_obj.encode("imagens/lena.jpg")
qrcode_obj.plot_bitplanes()
print("Texto extraído:", qrcode_obj.get_text())